# 04 · GGUF Export (optional)

**Phase notebook** — feature `004-notebook-split`. Consumes `lora_adapter/`; produces `gguf_export/` for local inference.
Run top-to-bottom on a fresh Colab runtime. All constants come from the shared `config.py`.

## Deviations
- **Notebook decomposition (constitution v1.2.0 §I / §Notebook Decomposition).** One of four phase
  notebooks split from the original single `notebook.ipynb`, for modularity/reuse and dependency-conflict
  isolation. Phases hand off **only** via Drive artifacts; the single shared `config.py` is the only
  cross-notebook import. This supersedes the single-notebook risk-first stage order (recorded per §Governance).


In [ ]:
# \u2500\u2500 Cell 1 \u00b7 Bootstrap: fetch shared config.py, install THIS phase's deps \u2500\u2500
# The four phase notebooks share exactly ONE module (config.py), fetched from the
# repo (constitution v1.2.0 \u00a7Notebook Decomposition). Set REPO_URL to your repo.
# For a PRIVATE repo, use a token URL (https://<TOKEN>@github.com/...) or the
# raw-download fallback shown below.
import os, sys, subprocess

REPO_URL = "https://github.com/YOUR_ORG/fine-tuning-demo.git"  # TODO: set to your repo
REPO_DIR = "/content/fine-tuning-demo"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
sys.path.insert(0, REPO_DIR)
# Fallback (config.py only, e.g. private repo without clone access):
#   from urllib.request import urlretrieve
#   urlretrieve("<raw-url>/config.py", "/content/config.py"); sys.path.insert(0, "/content")

import config  # the ONLY cross-notebook import (FR-012)

# Export phase: export tooling only.
PINNED_REQS = [
    "unsloth",
    "unsloth_zoo",
    "transformers>=5.2",
    "peft>=0.14",
]

config.mount_drive()
config.install_deps(PINNED_REQS)
config.set_seeds()

In [ ]:
# \u2500\u2500 Cell 2 \u00b7 Merge adapter + export GGUF (ported from monolith export cell) \u2500\u2500
import os, torch
from unsloth import FastLanguageModel
from peft import PeftModel

config.select_profile(config.detect_vram_gb())
ADAPTER_PATH = config.lora_adapter_path()

# Fail fast if the adapter is missing or was built for a different base (FR-007 / FR-013)
config.verify_meta(ADAPTER_PATH, {"base_model_id": config.MODEL_ID})

# Load base, attach adapter (injected into `model` in place), then merge-export
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=config.MODEL_ID, max_seq_length=config.MAX_SEQ_LEN, load_in_4bit=True, dtype=None)
peft_model = PeftModel.from_pretrained(model, ADAPTER_PATH)

GGUF_PATH = config.gguf_export_path()
os.makedirs(GGUF_PATH, exist_ok=True)
model.save_pretrained_merged(GGUF_PATH, tokenizer, save_method="merged_4bit")
print(f"\u2705 GGUF export saved to: {GGUF_PATH}")